# Student Report: 2D kMC Crystal Growth, GP Surrogate, and Constrained Optimization



## 1. Objective

The goal of this project is to:

1. simulate 2D crystal growth on a 100x100 triangular lattice using kinetic Monte Carlo (kMC),

2. generate a 200-sample dataset in a 4D process space,

3. train Gaussian Process (GP) surrogate models for coverage and grain boundary density (GBD),

4. find growth conditions that minimize GBD while maintaining coverage above 80%.



## 2. Method Summary

### 2.1 kMC simulator

The simulator includes three events: adsorption, desorption, and diffusion. Grain IDs are tracked so that grain boundary density can be measured from neighboring occupied sites with different grain labels.



### 2.2 Parameter space and sampling

A Latin Hypercube design is used over 4 process variables:

- F in [0.01, 1.0]

- E_d in [0.1, 0.8]

- E_des in [1.0, 3.0]

- T in [200, 800]



### 2.3 Surrogate modeling

Two GP regressors are trained:

- GP_coverage predicts final surface coverage

- GP_gbd predicts final grain boundary density



Input preprocessing uses log10 transform on F, then standard scaling. Model quality is checked with 5-fold cross-validation.



### 2.4 Optimization strategy

Constrained surrogate optimization is applied to minimize predicted GBD under the constraint coverage >= 0.8. The optimizer report is saved in models/bo_result.json.


## 3. Results

### 3.1 Dataset snapshot (current artifact)

Using the current stored dataset artifact (data/dataset_old.npz):

- Number of simulations: 200

- Coverage range: 0.0 to 1.0

- GBD range: 0.0 to 1.0



Observed trends (correlations):

- Coverage increases most with E_des (+0.5848)

- Coverage decreases with temperature T (-0.4970)

- GBD decreases with T (-0.5603)

- GBD increases with E_d (+0.5136)



### 3.2 GP surrogate quality

From saved model metadata:

- Coverage GP: RMSE = 0.0604, R2 = 0.9665

- GBD GP: RMSE = 0.0253, R2 = 0.9797



These metrics indicate strong interpolation performance on the current data distribution.



### 3.3 Constrained optimization output

From models/bo_result.json, one best candidate is:

- F = 0.992452

- E_d = 0.102998

- E_des = 2.464734

- T = 225.525



Predicted at that point:

- Coverage = 0.9689 +/- 0.0485

- GBD = 0.0000 +/- 0.0558

- P(coverage >= 0.8) = 0.9998



## 4. Discussion

Important limitations:

1. The simulation is currently not working as intended because the current parameter bounds are likely mis-specified and bias many runs toward high coverage. This means the dataset is not fully representative of the physical behavior we want to model.

2. Dataset generation is computationally expensive and currently takes a long time (about 4 to 5 hours for a full run), which slows down iteration and retuning.



Because of these issues, BO recommendations should be interpreted carefully. If most sampled conditions are already high-coverage, the surrogate can become overconfident in that region and underestimate uncertainty in less-sampled regimes.


## 5. Simulation Visualization & Physical Constraints

To better analyze the crystalline growth visually and diagnose why some parameter combinations yield specific Grain Boundary Densities (GBD), a matplotlib-based animation pipeline has been introduced.

### The Kinetic Stability Rule
Visualizing earlier simulations revealed that highly coordinated atoms could unphysically continue to diffuse or detach. To correct this, a strict, instantaneous stability rule has been integrated directly into the `run_kmc` mathematics:
* **Rule**: By default, any atom that achieves $\ge 2$ occupied neighbors is completely locked into the lattice. Its probability to diffuse or desorb goes to $0$ until a neighbor leaves.
* This mandates permanent physical clustering and accurately portrays nucleus stabilization during growth.

You can render an MP4 or GIF of a single simulation to observe the clustering using `make_simulation_movie.py`:

In [ ]:
# Generate a short visualization of crystalline growth using the rigid stability constraints
# (Run this cell from the project root if scripts fail to locate each other)
!python ../make_simulation_movie.py --max-steps 15000 --fps 20 --snapshot-every 5 --out ../movies/stable_growth_demo.mp4

## 6. Full Pipeline Execution

Below are the commands to run the full data generation, surrogate, and optimization pipeline.

In [ ]:
!python ../run_batch.py

!python ../gp_surrogate.py

!python ../bayesian_optimize.py --coverage-target 0.80 --n-init 20 --n-iter 80 --n-candidates 5000 --seed 42 --bounds-source training

!python ../verify_dataset.py

## 7. Conclusion

The full workflow is implemented (kMC simulation, dataset generation, GP surrogate training, and constrained optimization), but the current setup should be treated as preliminary.

Main issues right now:
- parameter bounds need to be corrected/recalibrated so coverage is not systematically pushed high,
- dataset generation time (4 to 5 hours) makes experimentation slow.

After fixing bounds, the dataset should be regenerated and the GP + BO stages rerun.